<a href="https://colab.research.google.com/github/chetools/CHE4061_Spring2026/blob/main/PonchonSavarit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!wget -N -q https://raw.githubusercontent.com/chetools/chetools/main/tools/che5.ipynb -O che5.ipynb
%run che5.ipynb

In [14]:
p=Props(['Methanol','Isopropanol'])

In [15]:
def bubbleT_NRTL(x, P):

    T0 = np.sum(x*p.Tb(P))
    def froot(T):
        return np.sum(x*p.NRTL_gamma(x,T)*p.Pvap(T)/P)-1.

    T=sp.optimize.root_scalar(froot, x0=T0, method='secant').root
    return T, x*p.NRTL_gamma(x,T)*p.Pvap(T)/P

In [26]:
P=1e5
z1s = np.linspace(0,1,101)
y1s = []
bubbleTs = []
for x1 in x1s:
    T, (y1, _) = bubbleT_NRTL(np.array([x1, 1-x1]), P)
    bubbleTs.append(T)
    y1s.append(y1)
bubbleTs = np.array(bubbleTs)
y1s = np.array(y1s)

In [31]:
Ty_interp = sp.interpolate.PchipInterpolator(y1s, bubbleTs)
Tx_interp = sp.interpolate.PchipInterpolator(z1s, bubbleTs)

In [32]:
Hvs = [p.Hv([y1, 1-y1], Ty_interp(y1)) for y1 in z1s]
Hv_interp =sp.interpolate.PchipInterpolator(z1s, Hvs)

Hls = [p.Hl([x1, 1-x1], Tx_interp(x1)) for x1 in z1s]
Hl_interp =sp.interpolate.PchipInterpolator(z1s, Hls)


In [17]:
fig = make_subplots(rows=1,cols=2)
fig.add_scatter(x=x1s, y=bubbleTs, name='bubble', row=1,col=1)
fig.add_scatter(x=y1s, y=bubbleTs, name='dew', row=1,col=1)
fig.add_scatter(x=x1s, y=y1s, row=1, col=2, showlegend=False)
fig.add_scatter(x=[0,1],y=[0,1], row=1, col=2, mode='lines', showlegend=False, line_color='grey')
fig.update_layout(width=800, height=400)

In [ ]:
Hv(0.7)

In [ ]:
p.Hv([0.7,0.3], T)